# Lección 3 — Métricas de evaluación

**Tema:** cómo se mide si un modelo de Machine Learning es bueno o malo. **RMSE** y **R²** para regresión. **Confusion matrix** + **accuracy / precision / recall / F1** para clasificación.

**Objetivos de esta lección:**
- Visualizar un dataset antes de modelarlo.
- Interpretar el **RMSE** y el **R²** de un modelo de regresión en el contexto del problema.
- Leer una **confusion matrix** y entender los 4 cuadrantes (TP, FP, FN, TN).
- Calcular **accuracy, precision, recall y F1** a mano a partir de TP/FP/FN/TN.
- Elegir cuál métrica priorizar según el costo de falsos positivos vs falsos negativos.
- Detectar inequidad entre subgrupos cuando la accuracy global esconde fallas concentradas en un grupo.

**Side-note importante:** en este curso usamos los nombres de métricas en inglés — `accuracy` y `precision` son cosas DISTINTAS en ML, pero ambas se traducen como "precisión" al español. Para no confundirnos, usamos los nombres originales.

**Side-note importante 2:** este notebook usa `pandas`, `numpy`, `sklearn`, `matplotlib` y `seaborn`. Las vamos a aprender a usar más adelante. Por ahora el objetivo es leer los outputs e interpretar los resultados.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduPav/ai-learning-notebooks/blob/main/lessons/03-evaluation-metrics/notebook.ipynb)

## Índice

1. Regresión + **RMSE** — predecir alquileres a partir de la superficie del depto
2. Regresión + **R²** — qué tan bien explica el modelo la variabilidad
3. Clasificación + **Confusion Matrix** — detección de cáncer de mama
4. **Accuracy, Precision, Recall y F1**
5. La trampa del accuracy en datos desbalanceados
6. Cuando la accuracy global esconde inequidad entre grupos

In [ ]:
# Celda de setup: ejecutá esta celda una sola vez antes de avanzar.
# Todas las librerías vienen pre-instaladas en Google Colab,
# así que NO hace falta hacer pip install.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

# Fijamos seed para reproducibilidad.
np.random.seed(42)

# Estilo de las visualizaciones.
sns.set_theme(style="whitegrid")

print("Setup completo. Todo listo para correr las demos.")
print()
print("Nota: en este notebook calculamos TODAS las métricas a mano con numpy.")
print("Sin sklearn.metrics — para que veas exactamente qué están midiendo las fórmulas.")

## 1. Regresión + RMSE: predecir el alquiler de un departamento a partir de su superficie

**Idea central:** vamos a entrenar un modelo que predice el **alquiler mensual** de un departamento a partir de **un solo dato**: su superficie en metros cuadrados. Después aprendemos a MEDIR qué tan bueno es ese modelo usando una métrica llamada **RMSE**.

¿Por qué un dataset con UNA sola variable? Porque así podemos **graficarlo todo** en un scatter de 2D y _ver_ con los ojos qué hace el modelo y qué mide cada métrica.

**¿Qué es RMSE?** (Root Mean Squared Error — raíz del error cuadrático medio)

- Es un número que resume "qué tan lejos estuvieron las predicciones de la realidad".
- Fórmula:

`RMSE = raíz_cuadrada( promedio( (real - predicho)² ) )`
$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- ¿Por qué se eleva al cuadrado? Para **penalizar** los errores grandes más que los chicos. Un error de 10 cuenta 100; un error de 100 cuenta 10.000.
- ¿Por qué se toma raíz cuadrada al final? Para que el número quede en las mismas unidades del output (USD, no USD²).

**Lo importante:** RMSE bajo = modelo preciso. RMSE alto = modelo que se equivoca mucho. Pero "alto" o "bajo" depende del rango de los valores que predecís — un RMSE de 50 puede ser excelente o pésimo según de qué se trate.

In [ ]:
# Generamos un dataset sintético: 80 departamentos.
# Cada depto tiene una superficie en m² y un alquiler mensual en USD.
# Relación real (oculta): alquiler ≈ 8 USD por m² + 100 USD fijos + ruido.
# El modelo no conoce esa relación — la tiene que aprender de los datos.

rng = np.random.default_rng(42)
n = 80
superficie = rng.uniform(25, 120, size=n)              # entre 25 y 120 m²
ruido = rng.normal(0, 80, size=n)                      # ruido realista
alquiler = 8 * superficie + 100 + ruido                # USD por mes

df = pd.DataFrame({
    "superficie_m2": np.round(superficie, 1),
    "alquiler_USD": np.round(alquiler, 0),
})

print("Dimensiones del dataset:", df.shape)
df.head(8)

Cada fila es un departamento real. La columna `superficie_m2` es nuestra **feature** (input del modelo), y `alquiler_USD` es el **target** (lo que queremos predecir).

In [ ]:
# REGLA: antes de modelar, SIEMPRE visualizamos el dataset.
# Mirar la nube de puntos te dice si la relación tiene sentido,
# si hay outliers raros, y qué tan limpios o ruidosos son los datos.

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(df["superficie_m2"], df["alquiler_USD"], s=50, color="C0", alpha=0.7)
ax.set_xlabel("Superficie (m²)")
ax.set_ylabel("Alquiler mensual (USD)")
ax.set_title("Datos crudos: 80 departamentos")
plt.tight_layout()
plt.show()

**¿Qué se ve?** Una nube de puntos con tendencia clara: a más metros cuadrados, más alquiler. La relación parece **lineal** (los puntos se alinean alrededor de una recta imaginaria) pero con bastante **dispersión** — dos deptos del mismo tamaño pueden tener alquileres distintos. Eso es información valiosa: ya sabemos que un modelo lineal va a captar la tendencia, pero NO va a clavar las predicciones perfectamente. Va a errar.

In [ ]:
# Entrenamos un modelo de regresión lineal.
# Lo que el modelo "aprende" son dos números: pendiente y ordenada al origen
# para la mejor recta que atraviesa la nube.

X = df[["superficie_m2"]].values   # feature (en formato 2D que sklearn pide)
y = df["alquiler_USD"].values      # target

# Dividimos en train (70%) y test (30%) para evaluar honestamente.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Pendiente aprendida:        {modelo.coef_[0]:.2f} USD por m²")
print(f"Ordenada al origen:         {modelo.intercept_:.2f} USD")
print(f"(la relación real era: ~8 USD/m² + 100 USD fijos)")
print()

# Mostramos 10 deptos del test set: predicción vs real.
tabla = pd.DataFrame({
    "superficie_m2": X_test[:10, 0].round(1),
    "alquiler_real_USD": y_test[:10].round(0),
    "alquiler_predicho_USD": y_pred[:10].round(0),
    "error_USD": (y_pred[:10] - y_test[:10]).round(0),
})
tabla

In [ ]:
# Visualización: scatter + recta aprendida por el modelo.

xs = np.linspace(df["superficie_m2"].min(), df["superficie_m2"].max(), 100).reshape(-1, 1)
ys_line = modelo.predict(xs)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(X_train[:, 0], y_train, s=45, color="C0", alpha=0.6, label="train")
ax.scatter(X_test[:, 0], y_test, s=55, color="C1", alpha=0.9,
           edgecolor="black", linewidth=0.5, label="test")
ax.plot(xs, ys_line, color="C3", linewidth=2.5, label="recta del modelo")
ax.set_xlabel("Superficie (m²)")
ax.set_ylabel("Alquiler mensual (USD)")
ax.set_title("Recta aprendida por el modelo")
ax.legend()
plt.tight_layout()
plt.show()

**¿Qué mirar?** La recta roja es lo que el modelo "aprendió". Para cualquier superficie nueva, el modelo va a leer el alquiler que indica la recta. Los puntos cercanos a la recta = aciertos; los puntos lejos = errores.

In [ ]:
# Calculamos el RMSE A MANO (sin sklearn).
# Fórmula: raíz( promedio( (real - predicho)² ) )

errores = y_test - y_pred                       # diferencias punto a punto
errores_al_cuadrado = errores ** 2              # las elevamos al cuadrado
mse = errores_al_cuadrado.mean()                # promedio (Mean Squared Error)
rmse = np.sqrt(mse)                             # raíz cuadrada → RMSE

print(f"RMSE = USD {rmse:,.2f}")
print()
print(f"Rango de alquileres reales en test: USD {y_test.min():,.0f}  a  USD {y_test.max():,.0f}")
print(f"Amplitud del rango: USD {y_test.max() - y_test.min():,.0f}")
print(f"RMSE como % del rango: {rmse / (y_test.max() - y_test.min()) * 100:.1f}%")

**Interpretación:** el modelo se equivoca, en promedio, en algo así como **USD 90** por mes. Sobre alquileres que van de ~USD 300 a ~USD 1000, eso es ~12% del rango.

**Regla a recordar:** **RMSE NUNCA se interpreta solo.** Siempre se compara contra algo: rango, promedio o desviación estándar de la salida. Un RMSE de 50 puede ser excelente o pésimo según de qué se trate.

In [ ]:
# Visualización clave: qué ES un error, en el gráfico.
# Cada segmento gris vertical conecta un punto REAL con su predicción según el modelo.
# Esos segmentos son los "errores" que el RMSE resume en un solo número.

y_train_pred = modelo.predict(X_train)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_train[:, 0], y_train, s=45, color="C0", alpha=0.6, label="datos reales")
order = np.argsort(X_train[:, 0])
ax.plot(X_train[order, 0], y_train_pred[order],
        color="C3", linewidth=2.5, label="modelo (predicciones)")

# Dibujamos los segmentos verticales para ~20 puntos.
idx = np.random.RandomState(7).choice(len(X_train), 20, replace=False)
for i in idx:
    ax.plot([X_train[i, 0], X_train[i, 0]],
            [y_train[i], y_train_pred[i]],
            color="gray", linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_xlabel("Superficie (m²)")
ax.set_ylabel("Alquiler mensual (USD)")
ax.set_title("Cada segmento gris = un error de predicción\n(RMSE resume todos esos segmentos en un solo número)")
ax.legend()
plt.tight_layout()
plt.show()

**Lo que estás viendo es exactamente lo que mide RMSE:** la longitud de esos segmentos grises, elevados al cuadrado, promediados, y pasados por raíz cuadrada. Cuanto más cortos los segmentos en promedio, más bajo el RMSE.

In [ ]:
# ¿Por qué elevar los errores al cuadrado? Para penalizar más los errores grandes.
# Visualizamos: para 12 puntos, comparamos |error| (absoluto) vs error² (cuadrado).

errores_test = y_test - y_pred
# Tomamos 12 muestras ordenadas por |error| para ver el efecto del cuadrado.
idx_ordenado = np.argsort(np.abs(errores_test))
sample = idx_ordenado[::max(1, len(idx_ordenado) // 12)][:12]

abs_err = np.abs(errores_test[sample])
sq_err = errores_test[sample] ** 2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x_pos = np.arange(len(sample))

axes[0].bar(x_pos, abs_err, color="C0")
axes[0].set_title("|error|  (absoluto, en USD)")
axes[0].set_xlabel("muestra (ordenadas por error)")
axes[0].set_ylabel("USD")

axes[1].bar(x_pos, sq_err, color="C3")
axes[1].set_title("error²  (al cuadrado, en USD²)")
axes[1].set_xlabel("muestra (ordenadas por error)")
axes[1].set_ylabel("USD²")

plt.tight_layout()
plt.show()

**¿Qué se nota?** En el panel izquierdo (errores absolutos) las barras crecen suavemente. En el panel derecho (errores al cuadrado) las barras chicas casi no se ven y la última explota. **Ese es el efecto de elevar al cuadrado:** un error de 100 USD pesa 100× más que uno de 10 USD. Por eso RMSE es duro con los outliers — un modelo no puede "compensar" un error grande con muchos errores chicos.

### Ejercicio — interpretá el RMSE

**Contexto:** una empresa de delivery quiere estimarles a sus clientes cuántos minutos va a tardar su pedido. Entrenan un modelo que recibe la distancia y el nivel de tráfico, y devuelve los minutos estimados. En su ciudad los pedidos reales tardan entre **15 y 60 minutos**. El modelo entrenado tiene **RMSE = 6 minutos** en el set de testeo.

**Pregunta:** ¿es bueno ese RMSE? ¿Para qué casos de uso lo recomendarías y para cuáles **no**?

_Pista: ¿qué porcentaje del rango representa el error? ¿Cambia tu respuesta según para qué se use la predicción?_

<details>
<summary>Resultado esperado</summary>

El rango es de 45 minutos (60 − 15). RMSE = 6 minutos representa **~13% del rango** — un error moderado.

- **Sí lo recomendaría** para mostrarle al cliente una estimación general ("tu pedido llega en ~30 minutos"). Ahí 6 minutos de error es perfectamente tolerable.
- **NO lo recomendaría** para usos donde 6 minutos cambian la decisión: por ejemplo, coordinar la entrega con el horario exacto de una reunión, o disparar alertas automáticas si el repartidor se "demora" — la predicción podría disparar falsas alarmas o dejar pasar atrasos reales.

**Moraleja:** RMSE no se interpreta solo, ni contra un benchmark genérico. Se interpreta contra **lo que vas a hacer con la predicción**.

</details>

## 2. R² — qué tan bien explica el modelo la variabilidad

**Idea central:** el RMSE te da el error en las unidades del output (USD, minutos, kilos…).

**R²** ("R cuadrado") responde una pregunta distinta:

> _De toda la variabilidad que tiene el output, ¿qué fracción es la que mi modelo logra explicar?_

**Fórmula:**
$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$$

$$SS_{res} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
$$SS_{tot} = \sum_{i=1}^{n} (y_i - \bar{y})^2$$


donde:
- `SS_res` = suma de `(y - predicho)²` para todos los puntos → variabilidad que el modelo NO logra explicar.
- `SS_tot` = suma de `(y - promedio)²` para todos los puntos → variabilidad **total** del output.

**Interpretación:**
- **R² = 1.0** → el modelo explica TODA la variabilidad. Perfecto.
- **R² = 0.85** → el modelo explica el 85% de la variabilidad.
- **R² = 0.0** → el modelo no es mejor que predecir simplemente "el promedio" para todo.
- **R² < 0** → el modelo es PEOR que predecir el promedio. Algo está muy mal.

A diferencia del RMSE, R² es **adimensional** → te permite comparar modelos entre problemas distintos.

In [ ]:
# Calculamos R² A MANO (sin sklearn).
# Fórmula: 1 - SS_res / SS_tot

y_promedio = y_test.mean()
ss_tot = ((y_test - y_promedio) ** 2).sum()      # variabilidad total
ss_res = ((y_test - y_pred) ** 2).sum()          # variabilidad no explicada
r2 = 1 - ss_res / ss_tot

print(f"Promedio de los alquileres reales (test):  USD {y_promedio:,.2f}")
print(f"SS_tot (variabilidad total):                {ss_tot:,.0f}")
print(f"SS_res (variabilidad no explicada):         {ss_res:,.0f}")
print(f"R² = 1 - SS_res / SS_tot = {r2:.4f}")
print()
print(f"El modelo explica el {r2*100:.1f}% de la variabilidad de los alquileres.")

**Interpretación:** un R² alto (cercano a 1) significa que el modelo logra capturar la mayor parte del comportamiento del output. Si el R² es bajo, el modelo apenas le gana a "siempre predecir el promedio".

In [ ]:
# Visualización clave: dos paneles que muestran QUÉ son SS_tot y SS_res.
# Panel izquierdo: segmentos desde cada punto al PROMEDIO (eso es SS_tot).
# Panel derecho:   segmentos desde cada punto al MODELO (eso es SS_res).
# Visualmente: cuanto más cortos los segmentos del panel derecho vs el izquierdo,
# más alto el R² → más variabilidad explicada por el modelo.

xs_plot = np.linspace(X_test[:, 0].min(), X_test[:, 0].max(), 100).reshape(-1, 1)
ys_plot = modelo.predict(xs_plot)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

# Panel 1: SS_tot
axes[0].scatter(X_test[:, 0], y_test, s=55, color="C0", alpha=0.75)
axes[0].axhline(y_promedio, color="C2", linewidth=2.5,
                label=f"promedio = USD {y_promedio:,.0f}")
for i in range(len(X_test)):
    axes[0].plot([X_test[i, 0], X_test[i, 0]],
                 [y_test[i], y_promedio],
                 color="gray", linestyle="--", linewidth=0.9, alpha=0.7)
axes[0].set_title(f"Variabilidad total: SS_tot = {ss_tot:,.0f}\n(segmentos al promedio)")
axes[0].set_xlabel("Superficie (m²)")
axes[0].set_ylabel("Alquiler (USD)")
axes[0].legend()

# Panel 2: SS_res
axes[1].scatter(X_test[:, 0], y_test, s=55, color="C0", alpha=0.75)
axes[1].plot(xs_plot, ys_plot, color="C3", linewidth=2.5, label="modelo")
for i in range(len(X_test)):
    axes[1].plot([X_test[i, 0], X_test[i, 0]],
                 [y_test[i], y_pred[i]],
                 color="gray", linestyle="--", linewidth=0.9, alpha=0.7)
axes[1].set_title(f"Variabilidad no explicada: SS_res = {ss_res:,.0f}\n(segmentos al modelo)")
axes[1].set_xlabel("Superficie (m²)")
axes[1].legend()

plt.suptitle(f"R² = 1 - SS_res / SS_tot = {r2:.3f}    →    el modelo explica {r2*100:.1f}% de la variabilidad",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Lectura de los paneles:**

- **Panel izquierdo (SS_tot):** los segmentos grises miden cuánto se aleja cada punto real del **promedio**. Esa es TODA la variabilidad que el modelo "podría" explicar.
- **Panel derecho (SS_res):** los segmentos grises miden cuánto se aleja cada punto real de la **predicción del modelo**. Esa es la variabilidad que el modelo NO logró capturar.

**Cuanto más chicos (en suma) son los segmentos del panel derecho comparados con el izquierdo, más alto es el R².** Si los segmentos del panel derecho fueran cero, R² = 1 (modelo perfecto). Si fueran iguales a los del panel izquierdo, R² = 0 (el modelo no le gana al promedio).

### Ejercicio — interpretá un R²

a) Un modelo de regresión tiene **R² = 0.85**. ¿Qué porcentaje de la variabilidad del output explica?

b) Otro modelo, sobre datos distintos, da **R² = -0.3**. ¿Eso es siquiera posible? ¿Qué significa?

<details>
<summary>Resultado esperado</summary>

**a)** El modelo explica el **85%** de la variabilidad del output. Es un valor alto — el modelo capta bien el comportamiento de los datos.

**b)** Sí, es posible: R² puede ser negativo. Significa que el modelo predice **peor que simplemente devolver el promedio del target para cualquier input**. Es decir: te conviene tirar el modelo y reemplazarlo por una constante. Suele indicar un bug, datos mal preparados, o un modelo que no es apto para el problema (ej: usar regresión lineal cuando la relación es exponencial).

</details>

## 3. Clasificación + Confusion Matrix: detección de cáncer de mama

**Idea central:** ahora pasamos a **clasificación**. Vamos a entrenar un modelo que decide si un tumor es **maligno** o **benigno** a partir de mediciones de imágenes. Para evaluarlo NO basta con "¿acertó o no?" — necesitamos saber **QUÉ TIPO** de error comete. Para eso usamos la **confusion matrix**.

**¿Qué es una confusion matrix?**

Es una tabla de 2×2 (en clasificación binaria) que cruza lo que el modelo PREDIJO con lo que en realidad ERA. Sus 4 cuadrantes:

- **True Negatives (TN):** predijo benigno, ERA benigno. Acierto.
- **False Positives (FP):** predijo maligno, ERA benigno. Falsa alarma.
- **False Negatives (FN):** predijo benigno, ERA maligno. **Error grave en cáncer.**
- **True Positives (TP):** predijo maligno, ERA maligno. Acierto.

![Confusion Matrix](confusion-matrix.png)

**Importante:** los dos errores (FP y FN) NO son iguales. En cáncer, perder un caso real (FN) es mucho peor que asustar a un sano (FP).

In [ ]:
# Cargamos el dataset Breast Cancer Wisconsin (incluido en sklearn).
# Cada fila es un tumor; las features son medidas geométricas y de textura
# obtenidas de imágenes del tumor.
# El target en sklearn es: 0 = maligno, 1 = benigno.
# Lo invertimos para que 1 = "maligno" sea el caso POSITIVO (lo que nos importa detectar).

cancer = load_breast_cancer()
df_cancer = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df_cancer["clase"] = ["maligno" if t == 0 else "benigno" for t in cancer.target]

print("Dimensiones:", df_cancer.shape)
print()
print("Distribución de clases:")
print(df_cancer["clase"].value_counts())
print()
df_cancer[["mean radius", "mean texture", "mean perimeter", "mean area", "clase"]].head(6)

569 tumores en total, ~37% malignos y ~63% benignos. Hay 30 features por tumor (medidas de la imagen). La columna `clase` es el target.

In [ ]:
# REGLA: antes de entrenar, visualizamos. ¿Las features ayudan a distinguir
# malignos de benignos? Mostramos dos vistas que dan la respuesta:
# 1) Histograma de "mean radius" coloreado por clase.
# 2) Scatter de "mean radius" vs "mean texture" coloreado por clase.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: histograma de mean radius por clase.
for clase, color in [("benigno", "C0"), ("maligno", "C3")]:
    mask = df_cancer["clase"] == clase
    axes[0].hist(df_cancer.loc[mask, "mean radius"],
                 bins=25, alpha=0.6, label=clase, color=color)
axes[0].set_xlabel("mean radius")
axes[0].set_ylabel("cantidad de tumores")
axes[0].set_title("Distribución de 'mean radius' por clase")
axes[0].legend()

# Panel 2: scatter mean radius vs mean texture.
for clase, color in [("benigno", "C0"), ("maligno", "C3")]:
    mask = df_cancer["clase"] == clase
    axes[1].scatter(df_cancer.loc[mask, "mean radius"],
                    df_cancer.loc[mask, "mean texture"],
                    color=color, alpha=0.5, label=clase, s=30)
axes[1].set_xlabel("mean radius")
axes[1].set_ylabel("mean texture")
axes[1].set_title("mean radius vs mean texture")
axes[1].legend()

plt.tight_layout()
plt.show()

**¿Qué se ve?** Los tumores **malignos** (rojo) tienden a tener `mean radius` MAYOR que los benignos (azul). En el histograma las dos distribuciones se separan, aunque hay zona de superposición. En el scatter de la derecha también se distinguen dos nubes. **Conclusión:** las features sí contienen señal — un modelo lineal debería poder aprender una buena separación. Y como hay zona de superposición, sabemos de antemano que el modelo va a cometer algunos errores.

In [ ]:
# Entrenamos un clasificador (Logistic Regression).
# Convención: 1 = maligno (el caso "positivo"), 0 = benigno.

X = cancer.data
y = (cancer.target == 0).astype(int)   # 1 si maligno, 0 si benigno

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Estandarizamos features para que LogisticRegression converja mejor.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_s, y_train)
y_pred = clf.predict(X_test_s)

print(f"Tamaño del set de testeo: {len(y_test)} tumores")
print(f"Predicciones generadas:   {len(y_pred)}")

In [ ]:
# Construimos la confusion matrix A MANO con máscaras booleanas.
# Cada cuadrante es un conteo simple de cuándo coinciden (o no) y_test e y_pred.

tp = int(((y_test == 1) & (y_pred == 1)).sum())   # ambos = maligno
tn = int(((y_test == 0) & (y_pred == 0)).sum())   # ambos = benigno
fp = int(((y_test == 0) & (y_pred == 1)).sum())   # real benigno, predicho maligno
fn = int(((y_test == 1) & (y_pred == 0)).sum())   # real maligno, predicho benigno

# Layout estándar: filas = real, columnas = predicho. [BENIGNO, MALIGNO].
cm = np.array([[tn, fp],
               [fn, tp]])

print(f"True Negatives  (TN): {tn} → predijo benigno, ERA benigno")
print(f"False Positives (FP): {fp} → predijo maligno, ERA benigno (falsa alarma)")
print(f"False Negatives (FN): {fn} → predijo benigno, ERA maligno (ERROR GRAVE)")
print(f"True Positives  (TP): {tp} → predijo maligno, ERA maligno")
print()
print("Total de tumores en test:", tn + fp + fn + tp)

In [ ]:
# Visualizamos la confusion matrix como heatmap (mucho más legible).

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicho: BENIGNO", "Predicho: MALIGNO"],
    yticklabels=["Real: BENIGNO", "Real: MALIGNO"],
    cbar=False, annot_kws={"size": 22}, ax=ax,
)
ax.set_title("Confusion Matrix — detección de cáncer", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

**¿Qué mirar en este heatmap?**

- **Diagonal principal** (arriba-izq y abajo-der): aciertos del modelo. Números grandes = bueno.
- **Anti-diagonal** (arriba-der y abajo-izq): errores. Números chicos = bueno.
- **Cuadrante crítico:** abajo-izquierda → era maligno y el modelo dijo benigno. Estos son los **False Negatives**, los pacientes que se irían a casa pensando que están sanos cuando NO lo están.

### Ejercicio — interpretá la confusion matrix

En este caso del cáncer, ¿qué es peor: un FP o un FN? ¿Por qué?

<details>
<summary>Resultado esperado</summary>

Un **FN** (False Negative) es peor: el modelo predijo "benigno" pero el tumor era maligno. Ese paciente se iría a casa sin tratamiento cuando necesita atención urgente. Un FP (falsa alarma) también tiene costo — estrés y más estudios — pero se detecta con un chequeo adicional. En medicina, el principio es: mejor errar del lado de la precaución.

</details>

**Importante:** la confusion matrix es lo CRUDO. No te dice "el modelo es 90% bueno". Te dice exactamente cuántos errores de cada tipo cometió. Para resumirla en un número usamos las 4 métricas que vienen ahora.

## 4. Derivar accuracy, precision, recall y F1-score desde la confusion matrix

**Idea central:** las 4 métricas que más vas a ver en clasificación (accuracy, precision, recall, F1) son distintas formas de RESUMIR la confusion matrix en UN solo número. Cada una pone el foco en una pregunta diferente.

Vamos una por una. **Las calculamos a mano con las fórmulas — sin sklearn.**

### 4.1. Accuracy — "de todas las predicciones, ¿cuántas estuvieron bien?"

Fórmula: `(TP + TN) / (TP + TN + FP + FN)`

Es la métrica más intuitiva: aciertos sobre total. **Pero tiene una trampa famosa que vamos a ver al final.**

In [ ]:
# Accuracy = aciertos / total
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"Accuracy = (TP + TN) / total")
print(f"         = ({tp} + {tn}) / {tp + tn + fp + fn}")
print(f"         = {accuracy:.4f}")
print()
print(f"Interpretación: de cada 100 tumores, el modelo acierta {accuracy*100:.1f}.")

### 4.2. Precision — "de los que el modelo predijo positivos, ¿cuántos eran realmente positivos?"

Fórmula: `TP / (TP + FP)`

Pregunta sobre **las predicciones positivas del modelo**. Si tu modelo dice "este tumor es maligno", ¿qué tan confiable es esa predicción?

Útil cuando el costo de un FALSO POSITIVO es alto (ej: filtros de spam — no querrías marcar un mail importante como spam).

In [ ]:
# Precision = TP / (TP + FP) — "cuando dije positivo, ¿cuántas veces acerté?"
precision = tp / (tp + fp)

print(f"Precision = TP / (TP + FP)")
print(f"          = {tp} / ({tp} + {fp})")
print(f"          = {precision:.4f}")
print()
print(f"Interpretación: cuando el modelo dijo 'maligno', acertó el {precision*100:.1f}% de las veces.")

### 4.3. Recall — "de los que ERAN positivos, ¿cuántos atrapamos?"

Fórmula: `TP / (TP + FN)`

Pregunta sobre **los positivos reales**. ¿Cuántos casos reales de cáncer atrapamos?

Útil cuando el costo de un FALSO NEGATIVO es alto (ej: detección de cáncer — no podemos perder un caso real).

In [ ]:
# Recall = TP / (TP + FN) — "de los positivos reales, ¿cuántos atrapé?"
recall = tp / (tp + fn)

print(f"Recall = TP / (TP + FN)")
print(f"       = {tp} / ({tp} + {fn})")
print(f"       = {recall:.4f}")
print()
print(f"Interpretación: de cada 100 tumores REALMENTE malignos, el modelo atrapó {recall*100:.1f}.")
print(f"Perdimos {(1-recall)*100:.1f}% de los cánceres reales.")

### 4.4. F1-score — "un balance entre precision y recall"

Fórmula: `2 * (precision * recall) / (precision + recall)` (esto se llama **media armónica**, NO es promedio simple).

F1 es alto solo si AMBAS (precision y recall) son altas. Si una de las dos es muy baja, F1 cae mucho. Útil cuando querés balance entre los dos tipos de error.

In [ ]:
# F1 = 2 * (P * R) / (P + R) — media armónica entre precision y recall.
f1 = 2 * (precision * recall) / (precision + recall)

print(f"F1 = 2 * (precision * recall) / (precision + recall)")
print(f"   = 2 * ({precision:.4f} * {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"   = {f1:.4f}")
print()
print("Si precision y recall son altos, F1 es alto.")
print("Si una de las dos cae, F1 lo refleja inmediatamente (a diferencia del promedio simple).")

### Resumen de las 4 métricas en este modelo

![alt text](confusion-matrix-metrics.png)

In [ ]:
# Tabla resumen de las 4 métricas calculadas a mano sobre el modelo de cáncer.

resumen = pd.DataFrame({
    "métrica": ["Accuracy", "Precision", "Recall", "F1"],
    "fórmula": [
        "(TP + TN) / total",
        "TP / (TP + FP)",
        "TP / (TP + FN)",
        "2*(P*R) / (P+R)",
    ],
    "valor": [accuracy, precision, recall, f1],
    "pregunta que responde": [
        "de todas las predicciones, ¿cuántas estuvieron bien?",
        "cuando dije positivo, ¿cuántas veces acerté?",
        "de los positivos reales, ¿cuántos atrapé?",
        "¿qué tan balanceado está el modelo?",
    ],
})
resumen.style.format({"valor": "{:.4f}"})

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_metric_focus(ax, mask, title, formula):
    bg = np.where(mask, 1.0, 0.0)
    ax.imshow(bg, cmap="YlOrRd", vmin=0, vmax=1.5, alpha=0.55)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    fontsize=20, fontweight="bold", color="black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred: MALIGNO", "Pred: BENIGNO"], fontsize=9)
    ax.set_yticklabels(["Real: MALIGNO", "Real: BENIGNO"], fontsize=9)
    ax.set_title(f"{title}\n{formula}", fontsize=11)

# cm layout: cm[0,0]=TP, cm[0,1]=FN, cm[1,0]=FP, cm[1,1]=TN.
mask_acc  = np.array([[True,  True ], [True, True ]])  # all cells for total
mask_prec = np.array([[True,  False], [True, False ]])  # columna predicho positivo (TP, FP)
mask_rec  = np.array([[True,  True ], [False, False]])  # fila real positivo (TP, FN)
mask_f1   = np.array([[True,  True ], [True,  False]])  # todo menos TN (TP, FN, FP)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
plot_metric_focus(axes[0], mask_acc,  "Accuracy",  "(TP + TN) / Total")
plot_metric_focus(axes[1], mask_prec, "Precision", "TP / (TP + FP)")
plot_metric_focus(axes[2], mask_rec,  "Recall",    "TP / (TP + FN)")
plot_metric_focus(axes[3], mask_f1,   "F1",        "balance entre P y R (todo menos TN)")
plt.tight_layout()
plt.show()

**Lectura de los 4 paneles:** cada métrica ilumina celdas distintas de la misma matriz.

- **Accuracy** mira la diagonal (todos los aciertos), comparada contra el total.
- **Precision** mira solo la columna "predicho positivo" — qué fracción de esa columna son aciertos.
- **Recall** mira solo la fila "real positivo" — qué fracción de los reales positivos atrapamos.
- **F1** combina precision y recall (toca todas las celdas excepto los TN).

**Por eso ninguna métrica reemplaza a las otras:** miran cosas distintas. Cuál priorizar depende del problema.

### Ejercicio integrador 1 — calculá las 4 métricas desde cero

Un modelo de spam clasifica 100 mails. La confusion matrix es:

| | Predicho: spam | Predicho: NO spam |
|---|---|---|
| **Real: spam** | 25 (TP) | 10 (FN) |
| **Real: NO spam** | 5 (FP) | 60 (TN) |

Calculá accuracy, precision y recall usando las fórmulas de esta sección.

<details>
<summary>Resultado esperado</summary>

- **Accuracy** = (25 + 60) / (25 + 60 + 5 + 10) = 85 / 100 = **0.85 (85%)**
- **Precision** = 25 / (25 + 5) = 25 / 30 ≈ **0.833 (83.3%)**
- **Recall** = 25 / (25 + 10) = 25 / 35 ≈ **0.714 (71.4%)**

El modelo tiene buena accuracy y precision, pero su recall indica que perdió el 28.6% de los spams reales — esos terminaron en la bandeja de entrada.

</details>

### Ejercicio integrador 2 — elegí la métrica correcta

Para cada uno de estos escenarios, decidí cuál métrica (accuracy, precision, recall o F1) priorizarías y por qué:

a) Un modelo que detecta si hay incendio en imágenes de cámaras de seguridad de un edificio.

b) Un modelo de recomendación de contenido que tiene que evitar tanto recomendar títulos irrelevantes como perderse títulos que te gustarían.

c) Un modelo que filtra currículums para una búsqueda de empleo muy competitiva (pocos candidatos calificados entre muchos).

<details>
<summary>Resultado esperado</summary>

**a) Detección de incendios:** priorizarías **recall**. El error grave es no detectar un incendio real (FN). Un FP (falsa alarma) molesta, pero un FN puede costar vidas.

**b) Recomendaciones de contenido:** priorizarías **F1**. Querés balance: no molestar al usuario con títulos malos (precision alta) y no perderte títulos que le gustarían (recall alto).

**c) Filtrado de currículums con pocos calificados:** priorizarías **precision**. Si los candidatos calificados son escasos, no querés desperdiciar tiempo de entrevistas en candidatos que no cumplen el perfil (FP alto). El costo de un FN también existe, pero es menor que el costo operativo de entrevistar a muchos candidatos inadecuados.

</details>

## 5. La trampa del 99% de accuracy en detección de fraude

Imaginemos que tenemos:

- **100 transacciones** en total.
- **99 son legítimas**, **1 es fraude** (caso extremadamente desbalanceado pero realista).
- Construimos un modelo "vago" que **siempre predice 'no es fraude'**, sin importar la transacción.

¿Cuánto da accuracy? ¿Cuánto da recall? Calculemos.

In [ ]:
# Caso fraude: el modelo dice "no fraude" para todas las transacciones.
# Convención: 1 = fraude (el positivo), 0 = legítima.

y_real_fraude = np.array([0]*99 + [1]*1)        # 99 legítimas + 1 fraude
y_pred_fraude = np.array([0]*100)                # el modelo dice "no fraude" a TODAS

# Confusion matrix a mano.
tp_f = int(((y_real_fraude == 1) & (y_pred_fraude == 1)).sum())
tn_f = int(((y_real_fraude == 0) & (y_pred_fraude == 0)).sum())
fp_f = int(((y_real_fraude == 0) & (y_pred_fraude == 1)).sum())
fn_f = int(((y_real_fraude == 1) & (y_pred_fraude == 0)).sum())

print(f"True Negatives  (TN): {tn_f}")
print(f"False Positives (FP): {fp_f}")
print(f"False Negatives (FN): {fn_f} ← el único fraude real, no lo atrapamos")
print(f"True Positives  (TP): {tp_f}")
print()

# Accuracy.
acc_f = (tp_f + tn_f) / (tp_f + tn_f + fp_f + fn_f)
print(f"Accuracy = ({tp_f} + {tn_f}) / 100 = {acc_f:.2%}  ← parece BUENÍSIMA")

# Recall: TP / (TP + FN).
recall_f = tp_f / (tp_f + fn_f) if (tp_f + fn_f) > 0 else float("nan")
print(f"Recall   = {tp_f} / ({tp_f} + {fn_f}) = {recall_f:.2%}  ← el modelo no atrapa NINGÚN fraude")

# Precision: TP / (TP + FP). En este caso es 0/0 → indefinido.
if (tp_f + fp_f) == 0:
    prec_f_str = "INDEFINIDO (no se puede dividir por 0)"
    prec_f_num = 0  # para el bar chart
else:
    prec_f_num = tp_f / (tp_f + fp_f)
    prec_f_str = f"{prec_f_num:.2%}"
print(f"Precision = {tp_f} / ({tp_f} + {fp_f}) = {prec_f_str}")

In [ ]:
# Visualización: las 3 métricas lado a lado revelan la trampa.
# Accuracy 99% al lado de Recall 0% deja claro que accuracy sola engaña.

metricas = ["Accuracy", "Recall", "Precision"]
valores  = [acc_f, recall_f, prec_f_num]
labels   = [f"{acc_f:.0%}", f"{recall_f:.0%}", "indefinido"]
colores  = ["C2", "C3", "lightgray"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metricas, valores, color=colores, edgecolor="black")
ax.set_ylim(0, 1.1)
ax.set_ylabel("valor")
ax.set_title("Modelo 'siempre digo NO fraude':\nla accuracy alta esconde un recall = 0",
             fontsize=12)
for bar, label in zip(bars, labels):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
            label, ha="center", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Conclusión clave:**

Un modelo que **NUNCA atrapa un fraude** tiene **99% de accuracy**. Es decir: la accuracy alta puede ser una mentira cuando los datos están desbalanceados. En este caso, **recall = 0** revela inmediatamente que el modelo es inútil para detectar fraude.

**Por eso accuracy ≠ precision, y por eso necesitamos las 4 métricas — no solo una.**

**Regla a recordar:** la métrica que uses depende del COSTO de cada tipo de error. No hay métrica universal.

## 6. Cuando la accuracy global esconde inequidad entre grupos

**Idea central:** un modelo puede tener una accuracy global razonable y, al mismo tiempo, rendir muy distinto en distintos subgrupos de la población. Si solo mirás el promedio, esa diferencia te pasa por arriba.

**El caso:** vamos a ver un modelo de detección de cáncer evaluado sobre 200 pacientes — 100 hombres y 100 mujeres. La accuracy global es del 92% (parece excelente). Pero al desagregar el **recall** por sexo aparece una brecha grande.

**Por qué importa:** en problemas con impacto humano (salud, crédito, contratación, justicia), una métrica global es un promedio que puede esconder fallas concentradas en un grupo. Antes de poner un modelo en producción, hay que desagregar las métricas por los subgrupos relevantes (sexo, edad, región, idioma, nivel socioeconómico, etc.).

In [ ]:
# Construimos la confusion matrix POR GRUPO usando los conteos del caso.
# Mismo patrón que en las secciones anteriores: TP/FN/FP/TN a mano.

# --- Hombres: 100 pacientes, 20 con cáncer (positivos), 80 sanos ---
tp_h, fn_h = 18, 2     # de 20 cánceres reales, atrapamos 18
fp_h, tn_h = 2, 78     # de 80 sanos, 2 falsa alarma, 78 bien clasificados

# --- Mujeres: 100 pacientes, 20 con cáncer, 80 sanas ---
tp_m, fn_m = 12, 8     # de 20 cánceres reales, atrapamos solo 12
fp_m, tn_m = 4, 76     # de 80 sanas, 4 falsa alarma, 76 bien clasificadas

# Métricas POR grupo
accuracy_h = (tp_h + tn_h) / (tp_h + tn_h + fp_h + fn_h)
recall_h   = tp_h / (tp_h + fn_h)

accuracy_m = (tp_m + tn_m) / (tp_m + tn_m + fp_m + fn_m)
recall_m   = tp_m / (tp_m + fn_m)

# Métricas GLOBALES (sumando los conteos de los dos grupos)
tp_g = tp_h + tp_m
tn_g = tn_h + tn_m
fp_g = fp_h + fp_m
fn_g = fn_h + fn_m

accuracy_global = (tp_g + tn_g) / (tp_g + tn_g + fp_g + fn_g)
recall_global   = tp_g / (tp_g + fn_g)

print("=== Métricas globales (todos los pacientes juntos) ===")
print(f"Accuracy global: {accuracy_global:.0%}   ← suena excelente")
print(f"Recall global:   {recall_global:.0%}")
print()
print("=== Desagregado por grupo ===")
print(f"Recall en hombres: {recall_h:.0%}")
print(f"Recall en mujeres: {recall_m:.0%}   ← 30 puntos menos")

**Lo que se ve:** la accuracy global del **92%** suena excelente. Pero al desagregar el recall por sexo:

- En **hombres**, el modelo atrapa el **90%** de los cánceres reales (pierde 1 de cada 10).
- En **mujeres**, el modelo atrapa solo el **60%** (pierde **4 de cada 10**).

Esa brecha de 30 puntos no aparece en ningún número global. Un médico que mire solo la accuracy del 92% no se entera de que su modelo es notablemente menos confiable en pacientes mujeres.

**¿Por qué pasa esto en la práctica?** Las causas más frecuentes son:
- **Datos de entrenamiento desbalanceados:** el modelo vio muchos más ejemplos de un grupo que del otro.
- **Features que se comportan distinto entre grupos:** las mismas mediciones pueden tener distinto significado clínico en hombres y mujeres, y el modelo aprendió la regla del grupo dominante.
- **Etiquetas de baja calidad en un subgrupo:** el dataset puede tener más errores de etiquetado en un grupo que en el otro.

In [ ]:
# Visualización en 3 paneles:
#  - Panel 1: bar chart de recall global vs por grupo (la brecha es invisible en el global).
#  - Paneles 2 y 3: confusion matrices de hombres y mujeres lado a lado, mismo formato que la sección 3.

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Panel 1: bar chart de recall.
metricas_g = ["Global", "Hombres", "Mujeres"]
valores_g  = [recall_global, recall_h, recall_m]
colores_g  = ["C0", "C2", "C3"]
bars = axes[0].bar(metricas_g, valores_g, color=colores_g, edgecolor="black")
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("recall")
axes[0].set_title("Recall global vs por grupo\n(la brecha NO aparece en el promedio)")
for bar, v in zip(bars, valores_g):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
                 f"{v:.0%}", ha="center", fontsize=13, fontweight="bold")

# Panel 2: confusion matrix HOMBRES.
cm_h = np.array([[tn_h, fp_h], [fn_h, tp_h]])
sns.heatmap(cm_h, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: BENIGNO", "Pred: MALIGNO"],
            yticklabels=["Real: BENIGNO", "Real: MALIGNO"],
            cbar=False, annot_kws={"size": 16}, ax=axes[1])
axes[1].set_title(f"Hombres — recall = {recall_h:.0%}")

# Panel 3: confusion matrix MUJERES.
cm_m = np.array([[tn_m, fp_m], [fn_m, tp_m]])
sns.heatmap(cm_m, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: BENIGNO", "Pred: MALIGNO"],
            yticklabels=["Real: BENIGNO", "Real: MALIGNO"],
            cbar=False, annot_kws={"size": 16}, ax=axes[2])
axes[2].set_title(f"Mujeres — recall = {recall_m:.0%}\n({fn_m} cánceres no detectados vs {fn_h} en hombres)")

plt.tight_layout()
plt.show()

**Regla a recordar — siempre desagregá las métricas por subgrupo:**

Así como en datos desbalanceados la accuracy puede mentir (sección 5), en poblaciones heterogéneas la accuracy global puede ocultar fallas concentradas en un grupo.

Cuando un modelo se va a aplicar a una población diversa, **siempre desagregá las métricas por los subgrupos relevantes** del problema (sexo, edad, región, idioma, nivel socioeconómico…). Si una métrica importante baja mucho en algún grupo, hay un problema de equidad que ningún número global te muestra.

Esto es la base de lo que en machine learning se llama **fairness** o **model bias** — un campo entero dedicado a detectar y corregir este tipo de inequidades antes de poner un modelo en producción.

### Ejercicio — desagregá un modelo de fraude por país

Un banco usa un modelo de detección de fraude. La empresa reporta una **accuracy global = 95%** y el equipo de producto está listo para poner el modelo en producción. Antes del go-live, alguien decide desagregar las métricas por país de residencia del cliente:

- **Recall en clientes locales:** 92%
- **Recall en clientes extranjeros:** 45%

Preguntas:

a) ¿Qué grupo está peor servido por el modelo y qué consecuencia tiene en la práctica?

b) ¿La accuracy global del 95% hubiera revelado este problema? ¿Qué deberían hacer antes de poner el modelo en producción?

<details>
<summary>Resultado esperado</summary>

**a)** Los **clientes extranjeros**: el modelo solo atrapa el 45% de sus fraudes — pierde más de la mitad. En la práctica esto significa exposición real al daño económico concentrada en un grupo: el banco les detecta menos fraudes y por lo tanto les protege peor su dinero. Los clientes locales están bien cubiertos (92% de recall).

**b)** **No.** La accuracy global del 95% es un promedio que el grupo dominante (clientes locales) infla. La inequidad estaba escondida.

Antes de producción habría que:
1. **Investigar la causa:** probablemente los datos de entrenamiento están desbalanceados (muchos más casos locales que extranjeros), o las features no transfieren bien entre poblaciones (ej. patrones de gasto distintos).
2. **Reentrenar** con datos más balanceados o features adicionales relevantes para clientes extranjeros.
3. Considerar **umbrales distintos por grupo** o un modelo separado para el grupo minoritario.
4. **Monitorear las métricas desagregadas en producción**, no solo el agregado.

</details>

## Resumen de la lección

- **Visualizá los datos primero.** Antes de modelar nada, mirá la nube de puntos, los histogramas o las distribuciones por clase. Eso te dice si el problema es lineal, si hay outliers, y qué tan limpios o ruidosos son los datos.

**Métricas de Regresión**
- **RMSE** mide qué tan lejos quedaron las predicciones de la realidad en regresión. Siempre se interpreta contra el rango del output. Penaliza muy mucho las predicciones muy erradas.
- **R²** mide qué fracción de la variabilidad logra explicar el modelo.

**Métricas de Clasificación**
- La **confusion matrix** muestra los 4 tipos de resultado en clasificación: TP, TN, FP, FN. Es lo CRUDO.
- **Accuracy / precision / recall / F1** son cuatro métricas distintas que resumen la confusion matrix en un número. Cada una responde una pregunta distinta.
  - **Accuracy:** de todas las predicciones, ¿cuántas estuvieron bien?
  - **Precision:** cuando dije positivo, ¿cuántas veces acerté?
  - **Recall:** de los positivos reales, ¿cuántos atrapé?
  - **F1:** balance entre precision y recall.

- En datasets desbalanceados, **accuracy puede mentir**. Hay que mirar precision y recall para ver la verdad.
- Las métricas globales pueden esconder **inequidad entre subgrupos**. Cuando el modelo se va a aplicar a una población diversa, evaluá las métricas por subgrupo (sexo, edad, región, etc.) y no solo el promedio global.
- La métrica a optimizar depende del **costo de cada tipo de error** en TU contexto específico.